# 00 - Baseline (the deliberately messy "before" picture)
Everything in one place: loading, cleaning, features, training, evaluation, saving.
Module 1 turns this notebook into the `prodml` package. **Commit it as-is.**

In [ ]:
import pickle
import os

import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# load one month of NYC green taxi trips (path is relative to this notebook)
df = pd.read_parquet('../data/green_tripdata_2019-08.parquet')
df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
df['lpep_dropoff_datetime'] = pd.to_datetime(df['lpep_dropoff_datetime'])
df.shape

In [ ]:
# target = trip duration in minutes, keep only 1..60 minute trips
df['duration'] = (df.lpep_dropoff_datetime - df.lpep_pickup_datetime).dt.total_seconds() / 60
df = df[(df.duration >= 1) & (df.duration <= 60)]
len(df)

In [ ]:
# feature: pickup-dropoff pair, e.g. '43_236'
df['PU_DO'] = df['PULocationID'].astype(str) + '_' + df['DOLocationID'].astype(str)

In [ ]:
# split by time: first 80% = train, last 20% = validation
df = df.sort_values('lpep_pickup_datetime').reset_index(drop=True)
cut = int(len(df) * 0.8)
df_train, df_val = df.iloc[:cut], df.iloc[cut:]
len(df_train), len(df_val)

In [ ]:
# vectorize + train
cols = ['PU_DO', 'trip_distance']
dv = DictVectorizer()
X_train = dv.fit_transform(df_train[cols].to_dict(orient='records'))
X_val = dv.transform(df_val[cols].to_dict(orient='records'))
y_train, y_val = df_train.duration.values, df_val.duration.values
lr = LinearRegression().fit(X_train, y_train)

In [ ]:
# evaluate  -> these two numbers go at the top of reports/module-1.md
y_pred = lr.predict(X_val)
print('Validation MAE :', round(mean_absolute_error(y_val, y_pred), 4))
print('Validation RMSE:', round(float(np.sqrt(mean_squared_error(y_val, y_pred))), 4))

In [ ]:
# save the fitted vectorizer + model together
os.makedirs('../models', exist_ok=True)
with open('../models/baseline.pkl', 'wb') as f:
    pickle.dump((dv, lr), f)